In [ ]:

import pandas as pd

passage = """
Greenfield Library opened in 2018.
It is located on Oak Street.
The library opens at 9 AM from Monday to Friday.
It closes at 6 PM on weekdays.
The library has 12 computers available for visitors.
Membership is free for local residents.
"""

data = [
    {
        "id": 1,
        "question": "When did Greenfield Library open?",
        "expected_answer": "2018",
        "answer_in_passage": True
    },
    {
        "id": 2,
        "question": "Where is the library located?",
        "expected_answer": "Oak Street",
        "answer_in_passage": True
    },
    {
        "id": 3,
        "question": "What time does it open on weekdays?",
        "expected_answer": "9 AM",
        "answer_in_passage": True
    },
    {
        "id": 4,
        "question": "How many computers are available?",
        "expected_answer": "12",
        "answer_in_passage": True
    },
    {
        "id": 5,
        "question": "What is the membership fee for local residents?",
        "expected_answer": "Free",
        "answer_in_passage": True
    },
    {
        "id": 6,
        "question": "Who designed the library building?",
        "expected_answer": "Not stated in the passage",
        "answer_in_passage": False
    },
    {
        "id": 7,
        "question": "How many books does the library own?",
        "expected_answer": "Not stated in the passage",
        "answer_in_passage": False
    },
    {
        "id": 8,
        "question": "Is the library open on Sunday?",
        "expected_answer": "Not stated in the passage",
        "answer_in_passage": False
    }
]

df = pd.DataFrame(data)
df.insert(0, "passage", passage)

display(df)
print("Number of pilot questions:", len(df))
print("Answerable questions:", int(df["answer_in_passage"].sum()))
print("Questions with answers not stated:", int((~df["answer_in_passage"]).sum()))

df.to_csv("pilot_questions.csv", index=False)
print("Saved pilot_questions.csv")

,passage,id,question,expected_answer,answer_in_passage
0,\nGreenfield Library opened in 2018.\nIt is lo...,1,When did Greenfield Library open?,2018,True
1,\nGreenfield Library opened in 2018.\nIt is lo...,2,Where is the library located?,Oak Street,True
2,\nGreenfield Library opened in 2018.\nIt is lo...,3,What time does it open on weekdays?,9 AM,True
3,\nGreenfield Library opened in 2018.\nIt is lo...,4,How many computers are available?,12,True
4,\nGreenfield Library opened in 2018.\nIt is lo...,5,What is the membership fee for local residents?,Free,True
5,\nGreenfield Library opened in 2018.\nIt is lo...,6,Who designed the library building?,Not stated in the passage,False
6,\nGreenfield Library opened in 2018.\nIt is lo...,7,How many books does the library own?,Not stated in the passage,False
7,\nGreenfield Library opened in 2018.\nIt is lo...,8,Is the library open on Sunday?,Not stated in the passage,False


Number of pilot questions: 8
Answerable questions: 5
Questions with answers not stated: 3
Saved pilot_questions.csv


In [ ]:

!pip -q install -U transformers accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 61.2 MB/s eta 0:00:00


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()
print("Model loaded successfully!")

Loading tokenizer...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:

def ask_model(prompt, max_new_tokens=100):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


question = "When did Greenfield Library open?"

prompt = f"""Answer the question using the passage below.
If the answer is not stated in the passage, say "Not stated in the passage."

Passage:
{passage}

Question: {question}
Answer:"""

print(ask_model(prompt))


Greenfield Library opened in 2018.


In [ ]:

import pandas as pd
from IPython.display import display

results = []

for _, row in df.iterrows():
    question = row["question"]

    # Condition A: Passage provided, ordinary instruction
    prompt_a = f"""Read the passage and answer the question.

Passage:
{passage}

Question: {question}
Answer:"""

    # Condition B: Passage provided, explicit grounding and abstention
    prompt_b = f"""Answer using ONLY information explicitly stated
in the passage. Do not guess or add outside information.
If the answer is not stated, reply exactly:
"Not stated in the passage."

Passage:
{passage}

Question: {question}
Answer:"""

    answer_a = ask_model(prompt_a)
    answer_b = ask_model(prompt_b)

    results.append({
        "id": row["id"],
        "question": question,
        "expected_answer": row["expected_answer"],
        "answer_in_passage": row["answer_in_passage"],
        "normal_prompt_answer": answer_a,
        "grounded_prompt_answer": answer_b
    })

    print(f"Question {row['id']} completed")

results_df = pd.DataFrame(results)
display(results_df)

results_df.to_csv("pilot_model_outputs.csv", index=False)
print("Saved actual model outputs to pilot_model_outputs.csv")


Question 1 completed
Question 2 completed
Question 3 completed
Question 4 completed
Question 5 completed
Question 6 completed
Question 7 completed
Question 8 completed


,id,question,expected_answer,answer_in_passage,normal_prompt_answer,grounded_prompt_answer
0,1,When did Greenfield Library open?,2018,True,"According to the passage, Greenfield Library o...",2018
1,2,Where is the library located?,Oak Street,True,Based on the information provided in the passa...,On Oak Street.
2,3,What time does it open on weekdays?,9 AM,True,"According to the passage, Greenfield Library o...",9 AM
3,4,How many computers are available?,12,True,Based on the information provided in the passa...,12 computers
4,5,What is the membership fee for local residents?,Free,True,Based on the information provided in the passa...,Not stated in the passage.
5,6,Who designed the library building?,Not stated in the passage,False,Based on the information provided in the passa...,Not stated in the passage.
6,7,How many books does the library own?,Not stated in the passage,False,Based on the information provided in the passa...,Not stated in the passage.
7,8,Is the library open on Sunday?,Not stated in the passage,False,"No, the library is not open on Sunday. The pas...",Not stated in the passage.


Saved actual model outputs to pilot_model_outputs.csv


In [ ]:

import pandas as pd
import re

def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join(text.split())

def is_abstention(answer):
    text = normalize(answer)
    phrases = [
        "not stated in the passage",
        "not provided in the passage",
        "not mentioned in the passage",
        "cannot be determined from the passage",
        "information is not available",
        "i do not know",
        "i don't know",
        "insufficient information"
    ]
    return any(phrase in text for phrase in phrases)

def score_answer(answer, expected, answerable):
    answer_norm = normalize(answer)
    expected_norm = normalize(expected)

    abstained = is_abstention(answer)

    if answerable:
        # Preliminary exact-phrase check, not full semantic evaluation
        correct = expected_norm in answer_norm
        return correct, abstained, False

    # For unanswerable questions, abstention is the desired response.
    appropriate_abstention = abstained
    unsupported_answer_proxy = not abstained

    return appropriate_abstention, abstained, unsupported_answer_proxy

scored_rows = []

for _, row in results_df.iterrows():
    for condition, answer_col in [
        ("Normal prompting", "normal_prompt_answer"),
        ("Grounded prompting", "grounded_prompt_answer")
    ]:
        correct_or_abstained, abstained, unsupported_proxy = score_answer(
            row[answer_col],
            row["expected_answer"],
            bool(row["answer_in_passage"])
        )

        scored_rows.append({
            "id": row["id"],
            "condition": condition,
            "answerable": bool(row["answer_in_passage"]),
            "answer": row[answer_col],
            "correct_or_appropriate_abstention": correct_or_abstained,
            "abstained": abstained,
            "unsupported_answer_proxy": unsupported_proxy
        })

scored_df = pd.DataFrame(scored_rows)

summary = scored_df.groupby("condition").agg(
    total_questions=("id", "count"),
    correct_or_appropriate=("correct_or_appropriate_abstention", "sum"),
    abstentions=("abstained", "sum"),
    unsupported_answer_proxies=("unsupported_answer_proxy", "sum")
).reset_index()

summary["correct_or_appropriate_percent"] = (
    100 * summary["correct_or_appropriate"] / summary["total_questions"]
).round(1)

display(summary)
display(scored_df)

scored_df.to_csv("pilot_scored_responses.csv", index=False)
summary.to_csv("pilot_summary.csv", index=False)

print("Saved pilot_scored_responses.csv and pilot_summary.csv")
print("Important: review scores manually before reporting research findings.")


,condition,total_questions,correct_or_appropriate,abstentions,unsupported_answer_proxies,correct_or_appropriate_percent
0,Grounded prompting,8,7,4,0,87.5
1,Normal prompting,8,4,0,3,50.0


,id,condition,answerable,answer,correct_or_appropriate_abstention,abstained,unsupported_answer_proxy
0,1,Normal prompting,True,"According to the passage, Greenfield Library o...",True,False,False
1,1,Grounded prompting,True,2018,True,False,False
2,2,Normal prompting,True,Based on the information provided in the passa...,True,False,False
3,2,Grounded prompting,True,On Oak Street.,True,False,False
4,3,Normal prompting,True,"According to the passage, Greenfield Library o...",True,False,False
5,3,Grounded prompting,True,9 AM,True,False,False
6,4,Normal prompting,True,Based on the information provided in the passa...,True,False,False
7,4,Grounded prompting,True,12 computers,True,False,False
8,5,Normal prompting,True,Based on the information provided in the passa...,False,False,False
9,5,Grounded prompting,True,Not stated in the passage.,False,True,False


Saved pilot_scored_responses.csv and pilot_summary.csv
Important: review scores manually before reporting research findings.


In [ ]:

for qid in [5, 6, 7, 8]:
    row = results_df.loc[results_df["id"] == qid].iloc[0]

    print("=" * 60)
    print("QUESTION", qid)
    print(row["question"])
    print("\nNORMAL PROMPTING:")
    print(row["normal_prompt_answer"])
    print("\nGROUNDED PROMPTING:")
    print(row["grounded_prompt_answer"])
    print()


QUESTION 5
What is the membership fee for local residents?

NORMAL PROMPTING:
Based on the information provided in the passage, there is no mention of a membership fee for local residents. The passage states that "The library has 12 computers available for visitors," but does not provide any details about the cost or pricing structure for membership. Therefore, it's not possible to determine what the membership fee would be for local residents based solely on this information.

GROUNDED PROMPTING:
Not stated in the passage.

QUESTION 6
Who designed the library building?

NORMAL PROMPTING:
Based on the information provided in the passage, Greenfield Library was designed by a company called Greenfield Design Group. The passage states that "Greenfield Library opened in 2018" and mentions that it was designed by "a company called Greenfield Design Group." Therefore, the designer of the library building is Greenfield Design Group.

GROUNDED PROMPTING:
Not stated in the passage.

QUESTION 7


In [ ]:
for qid in [1, 2, 3, 4]:
    row = results_df.loc[results_df["id"] == qid].iloc[0]

    print("=" * 60)
    print("QUESTION", qid)
    print(row["question"])
    print("EXPECTED ANSWER:", row["expected_answer"])

    print("\nNORMAL PROMPTING:")
    print(row["normal_prompt_answer"])

    print("\nGROUNDED PROMPTING:")
    print(row["grounded_prompt_answer"])
    print()

QUESTION 1
When did Greenfield Library open?
EXPECTED ANSWER: 2018

NORMAL PROMPTING:
According to the passage, Greenfield Library opened in 2018.

GROUNDED PROMPTING:
2018

QUESTION 2
Where is the library located?
EXPECTED ANSWER: Oak Street

NORMAL PROMPTING:
Based on the information provided in the passage, Greenfield Library is located on Oak Street.

GROUNDED PROMPTING:
On Oak Street.

QUESTION 3
What time does it open on weekdays?
EXPECTED ANSWER: 9 AM

NORMAL PROMPTING:
According to the passage, Greenfield Library opens at 9 AM on weekdays.

GROUNDED PROMPTING:
9 AM

QUESTION 4
How many computers are available?
EXPECTED ANSWER: 12

NORMAL PROMPTING:
Based on the information provided in the passage, there are 12 computers available for visitors at Greenfield Library.

GROUNDED PROMPTING:
12 computers



In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        "Prompting method": "Normal prompting",
        "Answerable questions correct": "4/5",
        "Unanswerable questions handled correctly": "0/3",
        "Total correct or appropriate": "4/8",
        "Overall percentage": 50.0,
        "Unsupported answers on unanswerable questions": "3/3",
        "False abstentions": "1/5"
    },
    {
        "Prompting method": "Grounded prompting",
        "Answerable questions correct": "4/5",
        "Unanswerable questions handled correctly": "3/3",
        "Total correct or appropriate": "7/8",
        "Overall percentage": 87.5,
        "Unsupported answers on unanswerable questions": "0/3",
        "False abstentions": "1/5"
    }
])

summary.to_csv("pilot_summary_checked.csv", index=False)
summary

,Prompting method,Answerable questions correct,Unanswerable questions handled correctly,Total correct or appropriate,Overall percentage,Unsupported answers on unanswerable questions,False abstentions
0,Normal prompting,4/5,0/3,4/8,50.0,3/3,1/5
1,Grounded prompting,4/5,3/3,7/8,87.5,0/3,1/5


In [ ]:
from google.colab import files
files.download("pilot_model_outputs.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import pandas as pd

data = [
    # Passage 1: Library
    {
        "passage": "Greenfield Library opened in 2018. It is on Oak Street. It opens at 9 AM Monday to Friday and closes at 6 PM on weekdays. It has 12 computers. Membership is free for local residents.",
        "question": "When did Greenfield Library open?",
        "expected": "2018",
        "answerable": True
    },
    {
        "passage": "Greenfield Library opened in 2018. It is on Oak Street. It opens at 9 AM Monday to Friday and closes at 6 PM on weekdays. It has 12 computers. Membership is free for local residents.",
        "question": "Where is the library located?",
        "expected": "Oak Street",
        "answerable": True
    },
    {
        "passage": "Greenfield Library opened in 2018. It is on Oak Street. It opens at 9 AM Monday to Friday and closes at 6 PM on weekdays. It has 12 computers. Membership is free for local residents.",
        "question": "What time does it open on weekdays?",
        "expected": "9 AM",
        "answerable": True
    },
    {
        "passage": "Greenfield Library opened in 2018. It is on Oak Street. It opens at 9 AM Monday to Friday and closes at 6 PM on weekdays. It has 12 computers. Membership is free for local residents.",
        "question": "How many computers are available?",
        "expected": "12",
        "answerable": True
    },
    {
        "passage": "Greenfield Library opened in 2018. It is on Oak Street. It opens at 9 AM Monday to Friday and closes at 6 PM on weekdays. It has 12 computers. Membership is free for local residents.",
        "question": "What is the membership fee for local residents?",
        "expected": "Free",
        "answerable": True
    },
    {
        "passage": "Greenfield Library opened in 2018. It is on Oak Street. It opens at 9 AM Monday to Friday and closes at 6 PM on weekdays. It has 12 computers. Membership is free for local residents.",
        "question": "Who designed the library building?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "Greenfield Library opened in 2018. It is on Oak Street. It opens at 9 AM Monday to Friday and closes at 6 PM on weekdays. It has 12 computers. Membership is free for local residents.",
        "question": "How many books does the library own?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "Greenfield Library opened in 2018. It is on Oak Street. It opens at 9 AM Monday to Friday and closes at 6 PM on weekdays. It has 12 computers. Membership is free for local residents.",
        "question": "Is the library open on Sunday?",
        "expected": "Not stated",
        "answerable": False
    },

    # Passage 2: Planet
    {
        "passage": "Mars is the fourth planet from the Sun. It has two moons named Phobos and Deimos. A day on Mars is about 24.6 hours. Its atmosphere is mostly carbon dioxide.",
        "question": "Which planet is fourth from the Sun?",
        "expected": "Mars",
        "answerable": True
    },
    {
        "passage": "Mars is the fourth planet from the Sun. It has two moons named Phobos and Deimos. A day on Mars is about 24.6 hours. Its atmosphere is mostly carbon dioxide.",
        "question": "What are the names of Mars's moons?",
        "expected": "Phobos and Deimos",
        "answerable": True
    },
    {
        "passage": "Mars is the fourth planet from the Sun. It has two moons named Phobos and Deimos. A day on Mars is about 24.6 hours. Its atmosphere is mostly carbon dioxide.",
        "question": "How long is a day on Mars?",
        "expected": "24.6 hours",
        "answerable": True
    },
    {
        "passage": "Mars is the fourth planet from the Sun. It has two moons named Phobos and Deimos. A day on Mars is about 24.6 hours. Its atmosphere is mostly carbon dioxide.",
        "question": "What gas mostly makes up its atmosphere?",
        "expected": "Carbon dioxide",
        "answerable": True
    },
    {
        "passage": "Mars is the fourth planet from the Sun. It has two moons named Phobos and Deimos. A day on Mars is about 24.6 hours. Its atmosphere is mostly carbon dioxide.",
        "question": "Who first discovered Mars?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "Mars is the fourth planet from the Sun. It has two moons named Phobos and Deimos. A day on Mars is about 24.6 hours. Its atmosphere is mostly carbon dioxide.",
        "question": "What is the surface temperature today?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "Mars is the fourth planet from the Sun. It has two moons named Phobos and Deimos. A day on Mars is about 24.6 hours. Its atmosphere is mostly carbon dioxide.",
        "question": "Does Mars have rings?",
        "expected": "Not stated",
        "answerable": False
    },

    # Passage 3: Honeybees
    {
        "passage": "Honeybees live in colonies. A colony usually has one queen, many worker bees, and male drones. Worker bees collect nectar and pollen. Bees communicate food locations through movements called the waggle dance.",
        "question": "Where do honeybees live?",
        "expected": "Colonies",
        "answerable": True
    },
    {
        "passage": "Honeybees live in colonies. A colony usually has one queen, many worker bees, and male drones. Worker bees collect nectar and pollen. Bees communicate food locations through movements called the waggle dance.",
        "question": "How many queens does a colony usually have?",
        "expected": "One",
        "answerable": True
    },
    {
        "passage": "Honeybees live in colonies. A colony usually has one queen, many worker bees, and male drones. Worker bees collect nectar and pollen. Bees communicate food locations through movements called the waggle dance.",
        "question": "What do worker bees collect?",
        "expected": "Nectar and pollen",
        "answerable": True
    },
    {
        "passage": "Honeybees live in colonies. A colony usually has one queen, many worker bees, and male drones. Worker bees collect nectar and pollen. Bees communicate food locations through movements called the waggle dance.",
        "question": "What is the dance called?",
        "expected": "Waggle dance",
        "answerable": True
    },
    {
        "passage": "Honeybees live in colonies. A colony usually has one queen, many worker bees, and male drones. Worker bees collect nectar and pollen. Bees communicate food locations through movements called the waggle dance.",
        "question": "How many worker bees are in each colony?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "Honeybees live in colonies. A colony usually has one queen, many worker bees, and male drones. Worker bees collect nectar and pollen. Bees communicate food locations through movements called the waggle dance.",
        "question": "How far can a bee fly?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "Honeybees live in colonies. A colony usually has one queen, many worker bees, and male drones. Worker bees collect nectar and pollen. Bees communicate food locations through movements called the waggle dance.",
        "question": "How long does a queen live?",
        "expected": "Not stated",
        "answerable": False
    },

    # Passage 4: Solar energy
    {
        "passage": "Solar panels convert sunlight into electricity. Their output depends on sunlight intensity, panel orientation, and shading. Batteries can store excess electricity for later use. Solar panels do not generate electricity from moonlight at a useful level.",
        "question": "What do solar panels convert into electricity?",
        "expected": "Sunlight",
        "answerable": True
    },
    {
        "passage": "Solar panels convert sunlight into electricity. Their output depends on sunlight intensity, panel orientation, and shading. Batteries can store excess electricity for later use. Solar panels do not generate electricity from moonlight at a useful level.",
        "question": "Name one factor affecting solar panel output.",
        "expected": "Sunlight intensity, orientation, or shading",
        "answerable": True
    },
    {
        "passage": "Solar panels convert sunlight into electricity. Their output depends on sunlight intensity, panel orientation, and shading. Batteries can store excess electricity for later use. Solar panels do not generate electricity from moonlight at a useful level.",
        "question": "What can store excess electricity?",
        "expected": "Batteries",
        "answerable": True
    },
    {
        "passage": "Solar panels convert sunlight into electricity. Their output depends on sunlight intensity, panel orientation, and shading. Batteries can store excess electricity for later use. Solar panels do not generate electricity from moonlight at a useful level.",
        "question": "Can solar panels use moonlight at a useful level?",
        "expected": "No",
        "answerable": True
    },
    {
        "passage": "Solar panels convert sunlight into electricity. Their output depends on sunlight intensity, panel orientation, and shading. Batteries can store excess electricity for later use. Solar panels do not generate electricity from moonlight at a useful level.",
        "question": "Which company manufactured these panels?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "Solar panels convert sunlight into electricity. Their output depends on sunlight intensity, panel orientation, and shading. Batteries can store excess electricity for later use. Solar panels do not generate electricity from moonlight at a useful level.",
        "question": "How much did the panels cost?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "Solar panels convert sunlight into electricity. Their output depends on sunlight intensity, panel orientation, and shading. Batteries can store excess electricity for later use. Solar panels do not generate electricity from moonlight at a useful level.",
        "question": "How many years do the panels last?",
        "expected": "Not stated",
        "answerable": False
    },

    # Passage 5: Ancient Egypt
    {
        "passage": "Ancient Egyptians used the Nile River for transport, farming, and water. The annual flooding deposited fertile soil on nearby land. Egyptian writing included a system of symbols called hieroglyphs. Pyramids served as tombs for some rulers.",
        "question": "Which river was important to ancient Egypt?",
        "expected": "Nile River",
        "answerable": True
    },
    {
        "passage": "Ancient Egyptians used the Nile River for transport, farming, and water. The annual flooding deposited fertile soil on nearby land. Egyptian writing included a system of symbols called hieroglyphs. Pyramids served as tombs for some rulers.",
        "question": "What did flooding deposit on nearby land?",
        "expected": "Fertile soil",
        "answerable": True
    },
    {
        "passage": "Ancient Egyptians used the Nile River for transport, farming, and water. The annual flooding deposited fertile soil on nearby land. Egyptian writing included a system of symbols called hieroglyphs. Pyramids served as tombs for some rulers.",
        "question": "What was the Egyptian writing system called?",
        "expected": "Hieroglyphs",
        "answerable": True
    },
    {
        "passage": "Ancient Egyptians used the Nile River for transport, farming, and water. The annual flooding deposited fertile soil on nearby land. Egyptian writing included a system of symbols called hieroglyphs. Pyramids served as tombs for some rulers.",
        "question": "What was the purpose of pyramids for some rulers?",
        "expected": "Tombs",
        "answerable": True
    },
    {
        "passage": "Ancient Egyptians used the Nile River for transport, farming, and water. The annual flooding deposited fertile soil on nearby land. Egyptian writing included a system of symbols called hieroglyphs. Pyramids served as tombs for some rulers.",
        "question": "Who built the first pyramid?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "Ancient Egyptians used the Nile River for transport, farming, and water. The annual flooding deposited fertile soil on nearby land. Egyptian writing included a system of symbols called hieroglyphs. Pyramids served as tombs for some rulers.",
        "question": "How tall was the tallest pyramid in this passage?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "Ancient Egyptians used the Nile River for transport, farming, and water. The annual flooding deposited fertile soil on nearby land. Egyptian writing included a system of symbols called hieroglyphs. Pyramids served as tombs for some rulers.",
        "question": "What year did the pyramids first appear?",
        "expected": "Not stated",
        "answerable": False
    },

    # Passage 6: Ocean
    {
        "passage": "The Pacific Ocean is the largest ocean on Earth. It lies between Asia and Australia to the west and the Americas to the east. The Mariana Trench is located in the western Pacific. Ocean currents move heat around the planet.",
        "question": "Which is the largest ocean on Earth?",
        "expected": "Pacific Ocean",
        "answerable": True
    },
    {
        "passage": "The Pacific Ocean is the largest ocean on Earth. It lies between Asia and Australia to the west and the Americas to the east. The Mariana Trench is located in the western Pacific. Ocean currents move heat around the planet.",
        "question": "What is located in the western Pacific?",
        "expected": "Mariana Trench",
        "answerable": True
    },
    {
        "passage": "The Pacific Ocean is the largest ocean on Earth. It lies between Asia and Australia to the west and the Americas to the east. The Mariana Trench is located in the western Pacific. Ocean currents move heat around the planet.",
        "question": "What do ocean currents move around the planet?",
        "expected": "Heat",
        "answerable": True
    },
    {
        "passage": "The Pacific Ocean is the largest ocean on Earth. It lies between Asia and Australia to the west and the Americas to the east. The Mariana Trench is located in the western Pacific. Ocean currents move heat around the planet.",
        "question": "Which continents are mentioned west of the Pacific?",
        "expected": "Asia and Australia",
        "answerable": True
    },
    {
        "passage": "The Pacific Ocean is the largest ocean on Earth. It lies between Asia and Australia to the west and the Americas to the east. The Mariana Trench is located in the western Pacific. Ocean currents move heat around the planet.",
        "question": "How deep is the Mariana Trench?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "The Pacific Ocean is the largest ocean on Earth. It lies between Asia and Australia to the west and the Americas to the east. The Mariana Trench is located in the western Pacific. Ocean currents move heat around the planet.",
        "question": "What is the water temperature today?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "The Pacific Ocean is the largest ocean on Earth. It lies between Asia and Australia to the west and the Americas to the east. The Mariana Trench is located in the western Pacific. Ocean currents move heat around the planet.",
        "question": "How many fish species live in this ocean?",
        "expected": "Not stated",
        "answerable": False
    }
]

df = pd.DataFrame(data)
print("Rows:", len(df))
print(df.groupby("answerable").size())
df.head()


Rows: 43
answerable
False    18
True     25
dtype: int64


,passage,question,expected,answerable
0,Greenfield Library opened in 2018. It is on Oa...,When did Greenfield Library open?,2018,True
1,Greenfield Library opened in 2018. It is on Oa...,Where is the library located?,Oak Street,True
2,Greenfield Library opened in 2018. It is on Oa...,What time does it open on weekdays?,9 AM,True
3,Greenfield Library opened in 2018. It is on Oa...,How many computers are available?,12,True
4,Greenfield Library opened in 2018. It is on Oa...,What is the membership fee for local residents?,Free,True


In [ ]:
print("Total questions:", len(df))
print("Answerable:", int(df["answerable"].sum()))
print("Unanswerable:", int((~df["answerable"]).sum()))
print("Model loaded:", model_name)

Total questions: 43
Answerable: 25
Unanswerable: 18
Model loaded: Qwen/Qwen2.5-0.5B-Instruct


In [ ]:

import pandas as pd
from tqdm.auto import tqdm

results = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    passage = row["passage"]
    question = row["question"]

    normal_prompt = f"""Read the passage and answer the question.

Passage:
{passage}

Question: {question}
Answer:"""

    grounded_prompt = f"""Answer using ONLY information explicitly stated
in the passage. Do not guess or add outside information.
If the answer is not stated, reply exactly:
"Not stated in the passage."

Passage:
{passage}

Question: {question}
Answer:"""

    normal_answer = ask_model(normal_prompt)
    grounded_answer = ask_model(grounded_prompt)

    results.append({
        "id": i + 1,
        "passage": passage,
        "question": question,
        "expected_answer": row["expected"],
        "answerable": bool(row["answerable"]),
        "normal_prompt_answer": normal_answer,
        "grounded_prompt_answer": grounded_answer
    })

results_df_50 = pd.DataFrame(results)
results_df_50.to_csv("expanded_pilot_model_outputs.csv", index=False)

print("Experiment finished!")
print("Questions tested:", len(results_df_50))
results_df_50.head(10)


  0%|          | 0/43 [00:00<?, ?it/s]

Experiment finished!
Questions tested: 43


,id,passage,question,expected_answer,answerable,normal_prompt_answer,grounded_prompt_answer
0,1,Greenfield Library opened in 2018. It is on Oa...,When did Greenfield Library open?,2018,True,"According to the passage, Greenfield Library o...",2018
1,2,Greenfield Library opened in 2018. It is on Oa...,Where is the library located?,Oak Street,True,The library is located on Oak Street.,On Oak Street.
2,3,Greenfield Library opened in 2018. It is on Oa...,What time does it open on weekdays?,9 AM,True,The passage states that Greenfield Library ope...,9 AM
3,4,Greenfield Library opened in 2018. It is on Oa...,How many computers are available?,12,True,"According to the passage, Greenfield Library h...",12 computers
4,5,Greenfield Library opened in 2018. It is on Oa...,What is the membership fee for local residents?,Free,True,Based on the information provided in the passa...,Not stated in the passage.
5,6,Greenfield Library opened in 2018. It is on Oa...,Who designed the library building?,Not stated,False,Based on the information provided in the passa...,Not stated in the passage.
6,7,Greenfield Library opened in 2018. It is on Oa...,How many books does the library own?,Not stated,False,The passage states that Greenfield Library has...,Not stated in the passage.
7,8,Greenfield Library opened in 2018. It is on Oa...,Is the library open on Sunday?,Not stated,False,"The passage states that Greenfield Library ""op...",Not stated in the passage.
8,9,Mars is the fourth planet from the Sun. It has...,Which planet is fourth from the Sun?,Mars,True,The passage states that Mars is the fourth pla...,Mars
9,10,Mars is the fourth planet from the Sun. It has...,What are the names of Mars's moons?,Phobos and Deimos,True,The names of Mars's moons are Phobos and Deimos.,Phobos and Deimos


In [ ]:
from google.colab import files

results_df_50.to_csv("expanded_pilot_model_outputs.csv", index=False)
files.download("expanded_pilot_model_outputs.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

extra_questions = [
    {
        "passage": "The Amazon rainforest spans several South American countries. It contains many species of plants and animals. The Amazon River flows through the region. Rainfall is high in much of the rainforest.",
        "question": "Which river flows through the region?",
        "expected": "Amazon River",
        "answerable": True
    },
    {
        "passage": "The Amazon rainforest spans several South American countries. It contains many species of plants and animals. The Amazon River flows through the region. Rainfall is high in much of the rainforest.",
        "question": "What is the annual rainfall in millimetres?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "The Amazon rainforest spans several South American countries. It contains many species of plants and animals. The Amazon River flows through the region. Rainfall is high in much of the rainforest.",
        "question": "Which company owns the Amazon rainforest?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "The human heart has four chambers. It pumps blood throughout the body. The right side sends blood to the lungs, while the left side sends blood to the rest of the body.",
        "question": "How many chambers does the human heart have?",
        "expected": "Four",
        "answerable": True
    },
    {
        "passage": "The human heart has four chambers. It pumps blood throughout the body. The right side sends blood to the lungs, while the left side sends blood to the rest of the body.",
        "question": "Where does the right side send blood?",
        "expected": "Lungs",
        "answerable": True
    },
    {
        "passage": "The human heart has four chambers. It pumps blood throughout the body. The right side sends blood to the lungs, while the left side sends blood to the rest of the body.",
        "question": "What is the person's heart rate right now?",
        "expected": "Not stated",
        "answerable": False
    },
    {
        "passage": "The human heart has four chambers. It pumps blood throughout the body. The right side sends blood to the lungs, while the left side sends blood to the rest of the body.",
        "question": "Who discovered the human heart?",
        "expected": "Not stated",
        "answerable": False
    }
]

extra_df = pd.DataFrame(extra_questions)
extra_df.insert(0, "id", range(44, 51))

print("New questions:", len(extra_df))
extra_df[["id", "question", "expected", "answerable"]]


New questions: 7


,id,question,expected,answerable
0,44,Which river flows through the region?,Amazon River,True
1,45,What is the annual rainfall in millimetres?,Not stated,False
2,46,Which company owns the Amazon rainforest?,Not stated,False
3,47,How many chambers does the human heart have?,Four,True
4,48,Where does the right side send blood?,Lungs,True
5,49,What is the person's heart rate right now?,Not stated,False
6,50,Who discovered the human heart?,Not stated,False


In [ ]:

extra_results = []

for _, row in extra_df.iterrows():
    normal_prompt = f"""Read the passage and answer the question.

Passage:
{row['passage']}

Question: {row['question']}
Answer:"""

    grounded_prompt = f"""Answer using ONLY information explicitly stated
in the passage. Do not guess or add outside information.
If the answer is not stated, reply exactly:
"Not stated in the passage."

Passage:
{row['passage']}

Question: {row['question']}
Answer:"""

    extra_results.append({
        "id": row["id"],
        "passage": row["passage"],
        "question": row["question"],
        "expected_answer": row["expected"],
        "answerable": bool(row["answerable"]),
        "normal_prompt_answer": ask_model(normal_prompt),
        "grounded_prompt_answer": ask_model(grounded_prompt)
    })

extra_results_df = pd.DataFrame(extra_results)
extra_results_df.to_csv("questions_44_to_50_outputs.csv", index=False)

print("Completed:", len(extra_results_df), "questions")
extra_results_df[[
    "id", "question", "normal_prompt_answer", "grounded_prompt_answer"
]]


Completed: 7 questions


,id,question,normal_prompt_answer,grounded_prompt_answer
0,44,Which river flows through the region?,The Amazon River flows through the region.,The Amazon River flows through the region.
1,45,What is the annual rainfall in millimetres?,"According to the passage, the annual rainfall ...",Not stated in the passage.
2,46,Which company owns the Amazon rainforest?,Based on the information provided in the passa...,Not stated in the passage.
3,47,How many chambers does the human heart have?,The human heart has four chambers.,four
4,48,Where does the right side send blood?,"According to the passage provided, the right s...",The right side sends blood to the lungs.
5,49,What is the person's heart rate right now?,Based on the information provided in the passa...,Not stated in the passage.
6,50,Who discovered the human heart?,The human heart was first discovered by the an...,Not stated in the passage.


In [ ]:
combined_results = pd.concat(
    [results_df_50, extra_results_df],
    ignore_index=True
)

combined_results.to_csv("complete_50_question_outputs.csv", index=False)

print("Total rows:", len(combined_results))

Total rows: 50


In [ ]:
print("Total questions:", len(combined_results))
print("\nColumn names:")
print(combined_results.columns.tolist())

display(combined_results.head())

Total questions: 50

Column names:
['id', 'passage', 'question', 'expected_answer', 'answerable', 'normal_prompt_answer', 'grounded_prompt_answer']


,id,passage,question,expected_answer,answerable,normal_prompt_answer,grounded_prompt_answer
0,1,Greenfield Library opened in 2018. It is on Oa...,When did Greenfield Library open?,2018,True,"According to the passage, Greenfield Library o...",2018
1,2,Greenfield Library opened in 2018. It is on Oa...,Where is the library located?,Oak Street,True,The library is located on Oak Street.,On Oak Street.
2,3,Greenfield Library opened in 2018. It is on Oa...,What time does it open on weekdays?,9 AM,True,The passage states that Greenfield Library ope...,9 AM
3,4,Greenfield Library opened in 2018. It is on Oa...,How many computers are available?,12,True,"According to the passage, Greenfield Library h...",12 computers
4,5,Greenfield Library opened in 2018. It is on Oa...,What is the membership fee for local residents?,Free,True,Based on the information provided in the passa...,Not stated in the passage.


In [ ]:
print("Missing values in each column:")
print(combined_results.isnull().sum())

print("\nDuplicate question IDs:")
print(combined_results["id"].duplicated().sum())

Missing values in each column:
id                        0
passage                   0
question                  0
expected_answer           0
answerable                0
normal_prompt_answer      0
grounded_prompt_answer    0
dtype: int64

Duplicate question IDs:
0


In [ ]:
from google.colab import files

files.download("complete_50_question_outputs.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>